<a href="https://colab.research.google.com/github/junseok-jay/AI_lab/blob/main/pipeline/model_export_IR2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Export model to IR

In [1]:
import os, torch
from torch import nn

# ====== 공통 설정 ======
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEQ = 16
BS  = 1

def show_export_ir(name: str, mod: nn.Module, example_args: tuple, example_kwargs: dict | None = None, max_lines=200):
    mod.eval()
    if example_kwargs is None:
        ep = torch.export.export(mod, example_args)
    else:
        ep = torch.export.export(mod, example_args, example_kwargs)

    g = str(ep.graph_module.graph)
    lines = g.splitlines()
    print(f"\n========== {name} (device={DEVICE}) ==========")
    print(f"Graph lines: {len(lines)} (showing last {min(max_lines, len(lines))})")
    print("\n".join(lines[-max_lines:]))

    return ep, example_args

In [2]:
# ====== HF 공통 유틸 ======
from transformers import AutoModel, AutoModelForCausalLM, AutoTokenizer

@torch.no_grad()
def rand_ids(vocab, bs=BS, seq=SEQ):
    return torch.randint(0, vocab, (bs, seq), device=DEVICE, dtype=torch.long)

@torch.no_grad()
def ones_mask(bs=BS, seq=SEQ):
    return torch.ones((bs, seq), device=DEVICE, dtype=torch.long)

In [8]:
# ====== 1) ResNet50 ======
def export_resnet50():
    import torchvision
    m = torchvision.models.resnet50(weights=None).to(DEVICE).eval()
    x = torch.randn(BS, 3, 224, 224, device=DEVICE)
    return show_export_ir("ResNet50", m, (x,), max_lines=500)

# ====== 실행 ======
ep_resnet50, x_resnet50 = export_resnet50()


========== ResNet50 (device=cuda) ==========
Graph lines: 498 (showing last 498)
graph():
    %p_conv1_weight : [num_users=1] = placeholder[target=p_conv1_weight]
    %p_bn1_weight : [num_users=1] = placeholder[target=p_bn1_weight]
    %p_bn1_bias : [num_users=1] = placeholder[target=p_bn1_bias]
    %p_layer1_0_conv1_weight : [num_users=1] = placeholder[target=p_layer1_0_conv1_weight]
    %p_layer1_0_bn1_weight : [num_users=1] = placeholder[target=p_layer1_0_bn1_weight]
    %p_layer1_0_bn1_bias : [num_users=1] = placeholder[target=p_layer1_0_bn1_bias]
    %p_layer1_0_conv2_weight : [num_users=1] = placeholder[target=p_layer1_0_conv2_weight]
    %p_layer1_0_bn2_weight : [num_users=1] = placeholder[target=p_layer1_0_bn2_weight]
    %p_layer1_0_bn2_bias : [num_users=1] = placeholder[target=p_layer1_0_bn2_bias]
    %p_layer1_0_conv3_weight : [num_users=1] = placeholder[target=p_layer1_0_conv3_weight]
    %p_layer1_0_bn3_weight : [num_users=1] = placeholder[target=p_layer1_0_bn3_weight]
  

In [14]:
from torch.export.graph_signature import InputSpec
# ====== 2) BERT-base-uncased (last_hidden_state Tensor만 반환) ======
class BertIRWrap(nn.Module):
    def __init__(self, bert):
        super().__init__()
        self.bert = bert
    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask, return_dict=True)
        return out.last_hidden_state  # [B, S, H]

def export_bert_base():
    model_id = "bert-base-uncased"
    bert = AutoModel.from_pretrained(model_id).to(DEVICE).eval()
    w = BertIRWrap(bert).to(DEVICE).eval()
    input_ids = rand_ids(bert.config.vocab_size)
    attn = ones_mask()
    return show_export_ir("BERT-base-uncased", w, (input_ids, attn))

# ====== 실행 ======
ep_bert, x_bert = export_bert_base()
print(x_bert)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



========== BERT-base-uncased (device=cuda) ==========
Graph lines: 513 (showing last 200)
    %transpose_12 : [num_users=1] = call_function[target=torch.ops.aten.transpose.int](args = (%view_9, 1, 2), kwargs = {})
    %linear_19 : [num_users=1] = call_function[target=torch.ops.aten.linear.default](args = (%layer_norm_6, %p_bert_encoder_layer_3_attention_self_key_weight, %p_bert_encoder_layer_3_attention_self_key_bias), kwargs = {})
    %view_10 : [num_users=1] = call_function[target=torch.ops.aten.view.default](args = (%linear_19, [1, 16, -1, 64]), kwargs = {})
    %transpose_13 : [num_users=1] = call_function[target=torch.ops.aten.transpose.int](args = (%view_10, 1, 2), kwargs = {})
    %linear_20 : [num_users=1] = call_function[target=torch.ops.aten.linear.default](args = (%layer_norm_6, %p_bert_encoder_layer_3_attention_self_value_weight, %p_bert_encoder_layer_3_attention_self_value_bias), kwargs = {})
    %view_11 : [num_users=1] = call_function[target=torch.ops.aten.view.default]

In [3]:
# ====== 3) GPT-2 (logits Tensor만 반환, use_cache=False) ======
class CausalLMLogitsWrap(nn.Module):
    def __init__(self, clm):
        super().__init__()
        self.clm = clm
        # 캐시 끄기(그래프 흔들림 방지)
        if hasattr(self.clm, "config"):
            self.clm.config.use_cache = False
        if hasattr(self.clm, "generation_config"):
            self.clm.generation_config.use_cache = False
    def forward(self, input_ids, attention_mask):
        out = self.clm(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
            return_dict=True,
        )
        return out.logits  # [B, S, V]

def export_gpt2():
    model_id = "gpt2"
    gpt2 = AutoModelForCausalLM.from_pretrained(model_id).to(DEVICE).eval()
    w_gpt2 = CausalLMLogitsWrap(gpt2).to(DEVICE).eval()
    input_ids = rand_ids(gpt2.config.vocab_size)
    attn = ones_mask()
    return show_export_ir("GPT-2", w_gpt2, (input_ids, attn))

# ====== 실행 ======
ep_gpt2, x_gpt2 = export_gpt2()
print(x_gpt2)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



========== GPT-2 (device=cuda) ==========
Graph lines: 675 (showing last 200)
    %layer_norm_14 : [num_users=1] = call_function[target=torch.ops.aten.layer_norm.default](args = (%add_29, [768], %p_clm_transformer_h_7_ln_1_weight, %p_clm_transformer_h_7_ln_1_bias), kwargs = {})
    %view_79 : [num_users=1] = call_function[target=torch.ops.aten.view.default](args = (%layer_norm_14, [-1, 768]), kwargs = {})
    %addmm_28 : [num_users=1] = call_function[target=torch.ops.aten.addmm.default](args = (%p_clm_transformer_h_7_attn_c_attn_bias, %view_79, %p_clm_transformer_h_7_attn_c_attn_weight), kwargs = {})
    %view_80 : [num_users=1] = call_function[target=torch.ops.aten.view.default](args = (%addmm_28, [1, 16, 2304]), kwargs = {})
    %split_7 : [num_users=3] = call_function[target=torch.ops.aten.split.Tensor](args = (%view_80, 768, 2), kwargs = {})
    %getitem_21 : [num_users=1] = call_function[target=operator.getitem](args = (%split_7, 0), kwargs = {})
    %getitem_22 : [num_users=1] =

In [6]:
# ====== 4) Llama 3.2 3B Instruct (logits Tensor만 반환, fp16 권장) ======
def export_llama32_3b():
    model_id = "meta-llama/Llama-3.2-3B-Instruct"

    # 토큰 필요하면 환경변수 HF_TOKEN 사용
    token = os.environ.get("HF_TOKEN", None)

    llama = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
        device_map=None,
        token=token,
    ).to(DEVICE).eval()

    w = CausalLMLogitsWrap(llama).to(DEVICE).eval()
    input_ids = rand_ids(llama.config.vocab_size)
    attn = ones_mask()

    return show_export_ir("Llama-3.2-3B-Instruct", w, (input_ids, attn))

# ====== 실행 ======
ep_llama, x_llama = export_llama32_3b()

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]


========== Llama-3.2-3B-Instruct (device=cuda) ==========
Graph lines: 2045 (showing last 200)
    %add_151 : [num_users=3] = call_function[target=torch.ops.aten.add.Tensor](args = (%add_149, %linear_174), kwargs = {})
    %_assert_tensor_metadata_default_110 : [num_users=0] = call_function[target=torch.ops.aten._assert_tensor_metadata.default](args = (%add_151,), kwargs = {dtype: torch.float16, device: cuda:0, layout: torch.strided})
    %to_110 : [num_users=2] = call_function[target=torch.ops.aten.to.dtype](args = (%add_151, torch.float32), kwargs = {})
    %pow_51 : [num_users=1] = call_function[target=torch.ops.aten.pow.Tensor_Scalar](args = (%to_110, 2), kwargs = {})
    %mean_50 : [num_users=1] = call_function[target=torch.ops.aten.mean.dim](args = (%pow_51, [-1], True), kwargs = {})
    %add_152 : [num_users=1] = call_function[target=torch.ops.aten.add.Tensor](args = (%mean_50, 1e-05), kwargs = {})
    %rsqrt_50 : [num_users=1] = call_function[target=torch.ops.aten.rsqrt.defaul

In [4]:
def test():
  from transformers import AutoModelForCausalLM
  from torch.distributed.pipelining import pipeline, SplitPoint

  model_id = "meta-llama/Llama-3.2-3B-Instruct"

  # # 토큰 필요하면 환경변수 HF_TOKEN 사용
  # token = os.environ.get("HF_TOKEN", None)

  llama = AutoModelForCausalLM.from_pretrained(
      model_id,
      torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
      device_map=None,
  ).to(DEVICE).eval()

  w = CausalLMLogitsWrap(llama).to(DEVICE).eval()
  input_ids = rand_ids(llama.config.vocab_size)
  attn = ones_mask()

  split = {f"clm.model.layers.6" : SplitPoint.END}

  pipe = pipeline(
      w,
      mb_args=(input_ids, attn),
      split_spec=split
  )
test()

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

AssertionError: 

# IR to Unflatten, split unflatten, troch.export

In [ ]:
import torch

def try_unflatten_exported(ep):
    try:
        from torch.export import unflatten
        m2 = unflatten(ep)   # 버전에 따라 unflatten(ep) 또는 unflatten(ep, ...) 형태
        print("✅ unflatten(exported) OK:", type(m2))
        return True, m2
    except Exception as e:
        print("❌ unflatten(exported) FAIL:", type(e), e)
        return False, None

In [ ]:
try_unflatten_exported(ep_resnet50)

In [ ]:
try_unflatten_exported(ep_bert)

In [ ]:
try_unflatten_exported(ep_gpt2)

In [ ]:
try_unflatten_exported(ep_llama)

# IR to split and unflatten

In [15]:
import torch
import traceback

def export_then_split_then_unflatten(ep, example_args, split_pred):
    # 1) export
    gm = ep.graph_module  # FX GraphModule

    # 2) FX split (그래프를 여러 submodule로 나눔)
    # split_pred(node) == True 이면 "여기서 경계" 성격
    from torch.fx.passes.split_module import split_module

    try:
        split_gm = split_module(gm, gm, split_pred)
        print("✅ split_module OK")
    except Exception as e:
        print("❌ split_module FAIL:", type(e), e)
        return None, None

    # 3) split된 각 submodule을 export->unflatten 해보기
    results = {}
    for name, sub in split_gm.named_children():
        try:
            ep_sub = torch.export.export(sub, example_args)
            from torch.export import unflatten
            _ = unflatten(ep_sub)
            results[name] = "OK"
        except Exception as e:
            results[name] = f"FAIL: {type(e).__name__}: {e}"
    return split_gm, results

In [16]:
def make_split_pred_contains(substr: str):
    def pred(node):
        # call_module 노드에서 target은 모듈 경로 문자열
        return (node.op == "call_module") and (substr in str(node.target))
    return pred

In [25]:
split_pred_resnet50 = make_split_pred_contains("layer2.0")
split_gm_resnet50, results_resnet50 = export_then_split_then_unflatten(ep_resnet50, (x_resnet50,), split_pred_resnet50)

for name, status in results_resnet50.items():
    print(f"- {name}: {status}")

✅ split_module OK
- submod_False: FAIL: TypeError: missing a required argument: 'p_conv1_weight'


In [18]:
split_pred_bert = make_split_pred_contains("bert.encoder.layer.6")
split_gm_bert, results_bert = export_then_split_then_unflatten(ep_bert, (x_bert,), split_pred_bert)

for name, status in results_bert.items():
    print(f"- {name}: {status}")

✅ split_module OK
- submod_False: FAIL: TypeError: missing a required argument: 'b_bert_embeddings_token_type_ids'


In [19]:
split_pred_gpt2 = make_split_pred_contains("clm.transformer.h.6")
split_gm_gpt2, results_gpt2 = export_then_split_then_unflatten(ep_gpt2, (x_gpt2,), split_pred_gpt2)

for name, status in results_gpt2.items():
    print(f"- {name}: {status}")

✅ split_module OK
- submod_False: FAIL: TypeError: missing a required argument: 'p_clm_lm_head_weight'


In [20]:
split_pred_llama = make_split_pred_contains("model.layers.6")
split_gm_llama, results_llama = export_then_split_then_unflatten(ep_llama, (x_llama,), split_pred_llama)

for name, status in results_llama.items():
    print(f"- {name}: {status}")

✅ split_module OK
- submod_1: FAIL: TypeError: missing a required argument: 'unsqueeze'
- submod_False: FAIL: TypeError: missing a required argument: 'input_ids'


# IR to

In [5]:
def print_targets(ep, contains=None, limit=80):
    ts=[]
    for n in ep.graph_module.graph.nodes:
        if n.op == "call_module":
            t=str(n.target)
            if contains is None or contains in t:
                ts.append(t)
    ts=sorted(set(ts))
    for t in ts[:limit]:
        print(t)

print_targets(ep_resnet50, "layer2")
print_targets(ep_bert, "encoder.layer")
print_targets(ep_gpt2, "transformer.h")
print_targets(ep_llama, "layers")

NameError: name 'ep_resnet50' is not defined

In [11]:
def node_stats(ep):
    from collections import Counter
    c = Counter(n.op for n in ep.graph_module.graph.nodes)
    print("node op counts:", dict(c))

node_stats(ep_resnet50)

node op counts: {'placeholder': 321, 'call_function': 175, 'output': 1}


In [10]:
def print_call_function_targets(ep, contains=None, limit=80):
    ts=set()
    for n in ep.graph_module.graph.nodes:
        if n.op == "call_function":
            t=str(n.target)
            if contains is None or contains in t:
                ts.add(t)
    ts=sorted(ts)
    print("call_function targets:", len(ts))
    for t in ts[:limit]:
        print(" ", t)

print_call_function_targets(ep_resnet50, contains="batch_norm")

call_function targets: 1
  aten.batch_norm.default


In [12]:
def sample_call_function_targets(ep, k=30):
    c = 0
    for n in ep.graph_module.graph.nodes:
        if n.op == "call_function":
            print(str(n.target))
            c += 1
            if c >= k:
                break
    print("printed", c)

sample_call_function_targets(ep_resnet50, 30)

aten.conv2d.default
aten.batch_norm.default
aten.relu_.default
aten.max_pool2d.default
aten.conv2d.default
aten.batch_norm.default
aten.relu_.default
aten.conv2d.default
aten.batch_norm.default
aten.relu_.default
aten.conv2d.default
aten.batch_norm.default
aten.conv2d.default
aten.batch_norm.default
aten.add_.Tensor
aten.relu_.default
aten.conv2d.default
aten.batch_norm.default
aten.relu_.default
aten.conv2d.default
aten.batch_norm.default
aten.relu_.default
aten.conv2d.default
aten.batch_norm.default
aten.add_.Tensor
aten.relu_.default
aten.conv2d.default
aten.batch_norm.default
aten.relu_.default
aten.conv2d.default
printed 30


In [13]:
import torch, traceback
from torch.fx.passes.split_module import split_module

def B_from_ep_call_function(ep, example_args, func_target: str):
    gm = ep.graph_module

    def pred(node):
        return (node.op == "call_function") and (str(node.target) == func_target)

    # 1) split
    split_gm = split_module(gm, gm, pred)
    print("✅ split_module OK. children:", [n for n, _ in split_gm.named_children()])

    # 2) export(split_gm)
    try:
        ep2 = torch.export.export(split_gm.eval(), example_args)
        print("✅ export(split_gm) OK")
    except Exception:
        print("❌ export(split_gm) FAIL")
        print("\n".join(traceback.format_exc().splitlines()[-60:]))
        return False

    # 3) unflatten(ep2)
    try:
        from torch.export import unflatten
        _ = unflatten(ep2)
        print("✅ unflatten(after split) OK")
        return True
    except Exception:
        print("❌ unflatten(after split) FAIL")
        print("\n".join(traceback.format_exc().splitlines()[-60:]))
        return False

In [16]:
B_from_ep_call_function(ep_resnet50, x_resnet50, "aten.conv2d.default")
# 또는
B_from_ep_call_function(ep_resnet50, x_resnet50, "aten.add_.Tensor")

RuntimeError: cycle exists between partitions!

In [17]:
import torch, traceback
from torch.fx.passes.split_module import split_module

def B_split_once_by_call_function(ep, example_args, func_target: str, which=1):
    gm = ep.graph_module
    hit = {"n": 0}

    def pred(node):
        if node.op == "call_function" and str(node.target) == func_target:
            hit["n"] += 1
            return hit["n"] == which   # ✅ N번째 매치에서만 split
        return False

    # 1) split
    split_gm = split_module(gm, gm, pred)
    print("✅ split_module OK (split at match #", which, ") children:",
          [n for n, _ in split_gm.named_children()])

    # 2) export(split_gm)
    ep2 = torch.export.export(split_gm.eval(), example_args)
    print("✅ export(split_gm) OK")

    # 3) unflatten
    try:
        from torch.export import unflatten
        _ = unflatten(ep2)
        print("✅ unflatten(after split) OK")
        return True
    except Exception:
        print("❌ unflatten(after split) FAIL")
        print("\n".join(traceback.format_exc().splitlines()[-60:]))
        return False

In [20]:
B_split_once_by_call_function(ep_resnet50, x_resnet50, "aten.conv2d.default", which=1)
# 안 되면 10번째, 50번째처럼 바꿔보면 됨
B_split_once_by_call_function(ep_resnet50, x_resnet50, "aten.conv2d.default", which=50)

✅ split_module OK (split at match # 1 ) children: ['submod_True', 'submod_False']


TypeError: missing a required argument: 'p_bn1_weight'

In [21]:
def B_split_at_kth_call_function(ep, example_args, k=200):
    gm = ep.graph_module
    idx = {"i": 0}

    def pred(node):
        if node.op == "call_function":
            idx["i"] += 1
            return idx["i"] == k
        return False

    split_gm = split_module(gm, gm, pred)
    print("✅ split_module OK at call_function #", k,
          "children:", [n for n, _ in split_gm.named_children()])

    ep2 = torch.export.export(split_gm.eval(), example_args)
    print("✅ export(split_gm) OK")

    from torch.export import unflatten
    _ = unflatten(ep2)
    print("✅ unflatten(after split) OK")
    return True

In [22]:
B_split_at_kth_call_function(ep_resnet50, x_resnet50, k=200)

✅ split_module OK at call_function # 200 children: ['submod_False']


TypeError: missing a required argument: 'p_bn1_weight'

In [24]:
import inspect
gm = ep_resnet50.graph_module
idx = {"i": 0}
k = 1
def pred(node):
        if node.op == "call_function":
            idx["i"] += 1
            return idx["i"] == k
        return False

split_gm = split_module(gm, gm, pred)
print(inspect.signature(split_gm.forward))
print([n.name for n in split_gm.graph.nodes if n.op=="placeholder"][:30])

(p_conv1_weight, p_bn1_weight, p_bn1_bias, p_layer1_0_conv1_weight, p_layer1_0_bn1_weight, p_layer1_0_bn1_bias, p_layer1_0_conv2_weight, p_layer1_0_bn2_weight, p_layer1_0_bn2_bias, p_layer1_0_conv3_weight, p_layer1_0_bn3_weight, p_layer1_0_bn3_bias, p_layer1_0_downsample_0_weight, p_layer1_0_downsample_1_weight, p_layer1_0_downsample_1_bias, p_layer1_1_conv1_weight, p_layer1_1_bn1_weight, p_layer1_1_bn1_bias, p_layer1_1_conv2_weight, p_layer1_1_bn2_weight, p_layer1_1_bn2_bias, p_layer1_1_conv3_weight, p_layer1_1_bn3_weight, p_layer1_1_bn3_bias, p_layer1_2_conv1_weight, p_layer1_2_bn1_weight, p_layer1_2_bn1_bias, p_layer1_2_conv2_weight, p_layer1_2_bn2_weight, p_layer1_2_bn2_bias, p_layer1_2_conv3_weight, p_layer1_2_bn3_weight, p_layer1_2_bn3_bias, p_layer2_0_conv1_weight, p_layer2_0_bn1_weight, p_layer2_0_bn1_bias, p_layer2_0_conv2_weight, p_layer2_0_bn2_weight, p_layer2_0_bn2_bias, p_layer2_0_conv3_weight, p_layer2_0_bn3_weight, p_layer2_0_bn3_bias, p_layer2_0_downsample_0_weight, p_l

In [25]:
import torch

def build_kwargs_for_split_gm(split_gm, orig_model, x_tensor):
    sd = orig_model.state_dict()  # weights + buffers 다 있음
    kwargs = {}
    for n in split_gm.graph.nodes:
        if n.op != "placeholder":
            continue
        name = n.name
        if name == "x":
            kwargs["x"] = x_tensor
            continue

        # p_ / b_ prefix 제거 후 key 복원
        if name.startswith("p_"):
            key = name[2:].replace("_", ".")
        elif name.startswith("b_"):
            key = name[2:].replace("_", ".")
        else:
            # 예상 외 placeholder면 일단 skip
            continue

        if key not in sd:
            raise KeyError(f"state_dict에 없음: placeholder='{name}' -> key='{key}'")

        t = sd[key]
        # device 맞추기
        kwargs[name] = t.to(x_tensor.device)
    return kwargs

In [27]:
# orig_model: export할 때 썼던 resnet 모듈 (state_dict 필요)
# x: 원래 입력
import torchvision
m = torchvision.models.resnet50(weights=None)
x = torch.randn(BS, 3, 224, 224, device=DEVICE)
kwargs = build_kwargs_for_split_gm(split_gm, m, x)

# split_gm 실행 가능
y = split_gm(**kwargs)

# 이제 이 kwargs를 example_kwargs로 export 가능
ep2 = torch.export.export(split_gm.eval(), (), kwargs)
from torch.export import unflatten
_ = unflatten(ep2)
print("✅ export+unflatten OK")

KeyError: "state_dict에 없음: placeholder='b_bn1_running_mean' -> key='bn1.running.mean'"

In [28]:
import torch

# resnet에서 언더스코어가 들어가는 "진짜 이름"들
KEEP_UNDERSCORE_TAIL = {
    "running_mean",
    "running_var",
    "num_batches_tracked",
}

def placeholder_to_statekey(name: str) -> str:
    # strip prefix
    assert name.startswith(("p_", "b_"))
    s = name[2:]  # remove p_/b_

    parts = s.split("_")

    # tail 처리: running_mean / running_var / num_batches_tracked
    if len(parts) >= 3:
        tail2 = "_".join(parts[-2:])
        tail3 = "_".join(parts[-3:])
        if tail3 in KEEP_UNDERSCORE_TAIL:
            head = parts[:-3]
            tail = tail3
        elif tail2 in KEEP_UNDERSCORE_TAIL:
            head = parts[:-2]
            tail = tail2
        else:
            head = parts[:-1]
            tail = parts[-1]
    else:
        head = parts[:-1]
        tail = parts[-1]

    # head는 '.'로, tail은 그대로(혹은 그냥 마지막 1개면 weight/bias라서 ok)
    key = ".".join(head + [tail])
    return key

def build_kwargs_for_split_gm(split_gm, orig_model, x_tensor):
    sd = orig_model.state_dict()
    kwargs = {}
    for n in split_gm.graph.nodes:
        if n.op != "placeholder":
            continue
        name = n.name
        if name == "x":
            kwargs["x"] = x_tensor
            continue
        if not (name.startswith("p_") or name.startswith("b_")):
            continue

        key = placeholder_to_statekey(name)
        if key not in sd:
            raise KeyError(f"state_dict에 없음: placeholder='{name}' -> key='{key}'")

        kwargs[name] = sd[key].to(x_tensor.device)
    return kwargs

In [29]:
import torchvision, torch

m = torchvision.models.resnet50(weights=None).to(DEVICE).eval()
x = torch.randn(1,3,224,224, device=DEVICE)

kwargs = build_kwargs_for_split_gm(split_gm, m, x)
print("kwargs size:", len(kwargs))
# 실행
y = split_gm(**kwargs)
print(type(y), getattr(y, "shape", None))

kwargs size: 321
<class 'tuple'> None


In [ ]:
import torch, traceback

def export_and_unflatten_split_gm(split_gm, kwargs):
    try:
        ep2 = torch.export.export(split_gm.eval(), (), kwargs)
        print("✅ export(split_gm) OK")
    except Exception:
        print("❌ export(split_gm) FAIL")
        print("\n".join(traceback.format_exc().splitlines()[-60:]))
        return False, None

    try:
        from torch.export import unflatten
        m2 = unflatten(ep2)
        print("✅ unflatten(ep2) OK:", type(m2))
        return True, m2
    except Exception:
        print("❌ unflatten(ep2) FAIL")
        print("\n".join(traceback.format_exc().splitlines()[-60:]))
        return False, None

ok, m2 = export_and_unflatten_split_gm(split_gm, kwargs)